## 导入包

In [ ]:
import tensorflow as tf
import numpy as np

## 定义拟合的函数，数据划分

In [145]:
# 1. 定义要拟合的自定义函数
def target_function(x):
    """自定义的目标函数：混合正弦和二次函数，具有非线性特征"""
    return np.sin(5 * x) * np.exp(-x**2) + 0.2 * x**2

# 2. 生成训练集和测试集
def generate_data():
    """
    返回格式：(train_data, test_data)
    train_data = (x_train, y_train), test_data = (x_test, y_test)
    """
    # 训练集：1000个样本，带噪声
    x_train = np.random.uniform(-2, 2, (1000, 1)).astype(np.float32)
    y_train = target_function(x_train) + np.random.normal(0, 0.02, x_train.shape).astype(np.float32)
    
    # 测试集：200个样本，无噪声
    x_test = np.linspace(-2, 2, 200).reshape(-1, 1).astype(np.float32)
    y_test = target_function(x_test).astype(np.float32)
    
    train_data = (x_train, y_train)
    test_data = (x_test, y_test)
    return train_data, test_data

## 模型构建

In [146]:
class myModel:
    def __init__(self):
        """声明模型参数（适配一维输入、一维输出）"""
        # 输入层到第一隐藏层：输入维度1，隐藏层64个神经元
        self.W1 = tf.Variable(
            initial_value=tf.random.normal(shape=(1, 64), stddev=0.1),  # 缩小初始化方差，加速收敛
            trainable=True,
            name='W1'
        )
        self.b1 = tf.Variable(
            initial_value=tf.zeros(shape=(64,)),
            trainable=True,
            name='b1'
        )
        
        # 第一隐藏层到第二隐藏层：64->32个神经元
        self.W2 = tf.Variable(
            initial_value=tf.random.normal(shape=(64, 32), stddev=0.1),
            trainable=True,
            name='W2'
        )
        self.b2 = tf.Variable(
            initial_value=tf.zeros(shape=(32,)),
            trainable=True,
            name='b2'
        )
        
        # 第二隐藏层到输出层：32->1（一维输出）
        self.W3 = tf.Variable(
            initial_value=tf.random.normal(shape=(32, 1), stddev=0.1),
            trainable=True,
            name='W3'
        )
        self.b3 = tf.Variable(
            initial_value=tf.zeros(shape=(1,)),
            trainable=True,
            name='b3'
        )

    def __call__(self, x):
        """实现模型前向传播，返回预测值（ReLU激活）"""
        # 输入层到第一隐藏层：ReLU激活
        hidden1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        
        # 第一隐藏层到第二隐藏层：ReLU激活
        hidden2 = tf.nn.relu(tf.matmul(hidden1, self.W2) + self.b2)
        
        # 输出层：无激活（回归任务）
        output = tf.matmul(hidden2, self.W3) + self.b3
        
        return output

## 计算 loss

In [147]:
@tf.function
def compute_loss(predictions, labels):
    return tf.reduce_mean(tf.square(predictions - labels))

@tf.function
def compute_rmse(predictions, labels):
    mse = tf.reduce_mean(tf.square(predictions - labels))
    rmse = tf.sqrt(mse)
    return rmse

@tf.function
def train_one_step(model, optimizer, x, y):
    """严格返回 (loss, rmse) 2个值"""
    with tf.GradientTape() as tape:
        predictions = model(x)
        loss = compute_loss(predictions, y)

    # 确保trainable_vars和梯度列表长度一致
    trainable_vars = [model.W1, model.b1, model.W2, model.b2, model.W3, model.b3]
    grads = tape.gradient(loss, trainable_vars)
    
    # 梯度更新
    optimizer.apply_gradients(zip(grads, trainable_vars))

    rmse = compute_rmse(predictions, y)
    
    return loss, rmse

@tf.function
def test(model, x, y):
    """严格返回 (loss, rmse) 2个值"""
    predictions = model(x)
    loss = compute_loss(predictions, y)
    rmse = compute_rmse(predictions, y)
    return loss, rmse

## 实际训练

In [148]:
train_data, test_data = generate_data()

model = myModel()
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

epochs = 800
for epoch in range(epochs):

    loss, rmse = train_one_step(model, optimizer, 
                                tf.constant(train_data[0], dtype=tf.float32), 
                                tf.constant(train_data[1], dtype=tf.float32))

    print('epoch', epoch+1, ': loss', loss.numpy(), '; rmse', rmse.numpy())
    
# 测试模型
loss, rmse = test(model, 
                  tf.constant(test_data[0], dtype=tf.float32), 
                  tf.constant(test_data[1], dtype=tf.float32))
print('\ntest loss', loss.numpy(), '; test rmse', rmse.numpy())

epoch 1 : loss 0.27823675 ; rmse 0.5274815
epoch 2 : loss 0.2740121 ; rmse 0.52346164
epoch 3 : loss 0.26977634 ; rmse 0.5194
epoch 4 : loss 0.26561102 ; rmse 0.51537466
epoch 5 : loss 0.2620417 ; rmse 0.51190007
epoch 6 : loss 0.2584935 ; rmse 0.50842255
epoch 7 : loss 0.2549438 ; rmse 0.5049196
epoch 8 : loss 0.25139683 ; rmse 0.50139487
epoch 9 : loss 0.24795087 ; rmse 0.49794665
epoch 10 : loss 0.24452958 ; rmse 0.49449933
epoch 11 : loss 0.24112372 ; rmse 0.4910435
epoch 12 : loss 0.23771758 ; rmse 0.4875629
epoch 13 : loss 0.23432189 ; rmse 0.48406807
epoch 14 : loss 0.23094268 ; rmse 0.48056495
epoch 15 : loss 0.2275762 ; rmse 0.47704947
epoch 16 : loss 0.2242334 ; rmse 0.4735329
epoch 17 : loss 0.22092633 ; rmse 0.470028
epoch 18 : loss 0.21765988 ; rmse 0.46654034
epoch 19 : loss 0.21445483 ; rmse 0.46309268
epoch 20 : loss 0.21133071 ; rmse 0.4597072
epoch 21 : loss 0.20830216 ; rmse 0.45640132
epoch 22 : loss 0.20533915 ; rmse 0.45314363
epoch 23 : loss 0.20244159 ; rmse 0.4